In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
print("*****Iniciando procesamiento de datos*****")
# 1. Definimos los archivos y sus regiones correspondientes
files_mapping = {
    'os_combined-af-monthly-200901-202604.csv': 'Africa',
    'os_combined-as-monthly-200901-202604.csv': 'Asia',
    'os_combined-eu-monthly-200901-202604.csv': 'Europe',
    'os_combined-na-monthly-200901-202604.csv': 'North America',
    'os_combined-oc-monthly-200901-202604.csv': 'Oceania',
    'os_combined-sa-monthly-200901-202604.csv': 'South America',
    'os_combined-ww-monthly-200901-202604.csv': 'World Wide'
}
dfs = []

# 2. Leemos cada archivo, agregamos la columna 'Region' y lo guardarmos en una lista
for filename, region in files_mapping.items():
    try:
        df_temp = pd.read_csv(filename)
        df_temp['Region'] = region
        dfs.append(df_temp)
        print(f"Cargado exitosamente: {filename} ({region})")
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {filename}")

*****Iniciando procesamiento de datos*****
Cargado exitosamente: os_combined-af-monthly-200901-202604.csv (Africa)
Cargado exitosamente: os_combined-as-monthly-200901-202604.csv (Asia)
Cargado exitosamente: os_combined-eu-monthly-200901-202604.csv (Europe)
Cargado exitosamente: os_combined-na-monthly-200901-202604.csv (North America)
Cargado exitosamente: os_combined-oc-monthly-200901-202604.csv (Oceania)
Cargado exitosamente: os_combined-sa-monthly-200901-202604.csv (South America)
Cargado exitosamente: os_combined-ww-monthly-200901-202604.csv (World Wide)


In [3]:
# 3. Concatenamos todos los DataFrames
# Alineamos las columnas. Si un OS existe en Asia pero no en África, se pondrá NaN.
df_combined = pd.concat(dfs, ignore_index=True)
df_combined.value_counts


<bound method DataFrame.value_counts of          Date  Windows  Android    iOS  Unknown  Series 40  OS X  SymbianOS  \
0     2009-01    97.65     0.00   0.04     0.61        0.0  0.50       0.98   
1     2009-02    97.74     0.00   0.04     0.48        0.0  0.55       0.93   
2     2009-03    97.58     0.00   0.07     0.50        0.0  0.56       1.03   
3     2009-04    97.38     0.00   0.08     0.47        0.0  0.59       1.17   
4     2009-05    97.24     0.00   0.06     0.50        0.0  0.62       1.16   
...       ...      ...      ...    ...      ...        ...   ...        ...   
1451  2025-12    30.07    38.89  15.63     7.29        0.0  3.51       0.00   
1452  2026-01    32.40    36.27  15.58     7.41        0.0  3.38       0.00   
1453  2026-02    31.12    36.02  17.08     7.85        0.0  3.43       0.00   
1454  2026-03    26.33    37.92  18.62     8.53        0.0  4.12       0.00   
1455  2026-04    29.03    36.32  18.00     8.81        0.0  3.73       0.00   

      Black

In [4]:
# 4. Transformamos de formato "Wide" (Columnas de OS) a formato "Long" (Una columna 'OS_Name' y una 'Market_Share')
# Identificamos las columnas que NO son sistemas operativos
id_vars = ['Date', 'Region']

df_long = df_combined.melt(
    id_vars=id_vars, 
    var_name='OS_Name', 
    value_name='Market_Share'
)

display(df_long)

,Date,Region,OS_Name,Market_Share
0,2009-01,Africa,Windows,97.65
1,2009-02,Africa,Windows,97.74
2,2009-03,Africa,Windows,97.58
3,2009-04,Africa,Windows,97.38
4,2009-05,Africa,Windows,97.24
...,...,...,...,...
37851,2025-12,World Wide,Nintendo 3DS,NaN
37852,2026-01,World Wide,Nintendo 3DS,NaN
37853,2026-02,World Wide,Nintendo 3DS,NaN
37854,2026-03,World Wide,Nintendo 3DS,NaN


In [5]:
# Rellenar los valores nulos (NaN) con 0, ya que significa 0% de cuota de mercado
df_long['Market_Share'] = df_long['Market_Share'].fillna(0).astype(float)
# Filtrar registros donde Market_Share es 0 (ayuda a reducir muchísimo el tamaño del dataset)
df_long = df_long[df_long['Market_Share'] > 0]
df_long.describe

<bound method NDFrame.describe of           Date         Region       OS_Name  Market_Share
0      2009-01         Africa       Windows         97.65
1      2009-02         Africa       Windows         97.74
2      2009-03         Africa       Windows         97.58
3      2009-04         Africa       Windows         97.38
4      2009-05         Africa       Windows         97.24
...        ...            ...           ...           ...
37136  2018-05  North America  Nintendo 3DS          0.01
37137  2018-06  North America  Nintendo 3DS          0.01
37138  2018-07  North America  Nintendo 3DS          0.01
37139  2018-08  North America  Nintendo 3DS          0.01
37142  2018-11  North America  Nintendo 3DS          0.01

[20038 rows x 4 columns]>

In [6]:
# 5. Enriquecer los datos: Categoría de Dispositivo
# Listas de categorización
desktop_os = ['Windows', 'OS X', 'macOS', 'Linux', 'Chrome OS']
mobile_os = ['Android', 'iOS', 'Series 40', 'SymbianOS', 'Samsung', 'BlackBerry OS', 
                'Nokia Unknown', 'bada', 'Tizen', 'LG', 'Sony Ericsson', 'KaiOS', 'Firefox OS', 'webOS']
console_os = ['Playstation', 'Xbox', 'Nintendo', 'Nintendo 3DS']
def classify_device(os_name):
    if os_name in desktop_os: return 'Desktop'
    if os_name in mobile_os: return 'Mobile'
    if os_name in console_os: return 'Console'
    return 'Unknown/Other' # Para "Unknown" o "Other"
    
df_long['Device_Category'] = df_long['OS_Name'].apply(classify_device)
display(df_long)


,Date,Region,OS_Name,Market_Share,Device_Category
0,2009-01,Africa,Windows,97.65,Desktop
1,2009-02,Africa,Windows,97.74,Desktop
2,2009-03,Africa,Windows,97.58,Desktop
3,2009-04,Africa,Windows,97.38,Desktop
4,2009-05,Africa,Windows,97.24,Desktop
...,...,...,...,...,...
37136,2018-05,North America,Nintendo 3DS,0.01,Console
37137,2018-06,North America,Nintendo 3DS,0.01,Console
37138,2018-07,North America,Nintendo 3DS,0.01,Console
37139,2018-08,North America,Nintendo 3DS,0.01,Console


In [7]:
# 6. Enriquecer los datos: Sistemas Activos vs Legacy
legacy_os = ['Series 40', 'SymbianOS', 'BlackBerry OS', 'Nokia Unknown', 
                'bada', 'Sony Ericsson', 'Firefox OS', 'webOS']
df_long['Is_Active'] = ~df_long['OS_Name'].isin(legacy_os)
display(df_long)


,Date,Region,OS_Name,Market_Share,Device_Category,Is_Active
0,2009-01,Africa,Windows,97.65,Desktop,True
1,2009-02,Africa,Windows,97.74,Desktop,True
2,2009-03,Africa,Windows,97.58,Desktop,True
3,2009-04,Africa,Windows,97.38,Desktop,True
4,2009-05,Africa,Windows,97.24,Desktop,True
...,...,...,...,...,...,...
37136,2018-05,North America,Nintendo 3DS,0.01,Console,True
37137,2018-06,North America,Nintendo 3DS,0.01,Console,True
37138,2018-07,North America,Nintendo 3DS,0.01,Console,True
37139,2018-08,North America,Nintendo 3DS,0.01,Console,True


In [8]:
# 7. Calcular el Índice de Concentración de Mercado (HHI) por Región y Fecha
# El HHI es la suma de los cuadrados de las cuotas de mercado de cada competidor.
df_long['Share_Squared'] = df_long['Market_Share'] ** 2

# Agrupamos para calcular el índice y lo renombramos
hhi_df = df_long.groupby(['Date', 'Region'])['Share_Squared'].sum().reset_index()
hhi_df.rename(columns={'Share_Squared': 'Market_Concentration_Index'}, inplace=True)

# Unimos el índice de vuelta a nuestro dataframe principal
df_final = pd.merge(df_long, hhi_df, on=['Date', 'Region'])

# Limpiamos columnas temporales y convertimos Date a formato DateTime
df_final.drop(columns=['Share_Squared'], inplace=True)
df_final['Date'] = pd.to_datetime(df_final['Date'])
display(df_final)

,Date,Region,OS_Name,Market_Share,Device_Category,Is_Active,Market_Concentration_Index
0,2009-01-01,Africa,Windows,97.65,Desktop,True,9537.1509
1,2009-02-01,Africa,Windows,97.74,Desktop,True,9554.5601
2,2009-03-01,Africa,Windows,97.58,Desktop,True,9523.5485
3,2009-04-01,Africa,Windows,97.38,Desktop,True,9484.8873
4,2009-05-01,Africa,Windows,97.24,Desktop,True,9457.7398
...,...,...,...,...,...,...,...
20033,2018-05-01,North America,Nintendo 3DS,0.01,Console,True,2708.5872
20034,2018-06-01,North America,Nintendo 3DS,0.01,Console,True,2775.2198
20035,2018-07-01,North America,Nintendo 3DS,0.01,Console,True,2783.4254
20036,2018-08-01,North America,Nintendo 3DS,0.01,Console,True,2791.8954


In [9]:
# 8. Guardar el nnuevo dataset procesado
output_filename = 'os_market_share_clean.csv'
df_final.to_csv(output_filename, index=False)

print("-" * 30)
print("¡Procesamiento terminado con éxito!")
print(f"Dataset final guardado como: {output_filename}")
print(f"Filas totales: {len(df_final)}")
print("Muestra de los datos:")
print(df_final.head())

------------------------------
¡Procesamiento terminado con éxito!
Dataset final guardado como: os_market_share_clean.csv
Filas totales: 20038
Muestra de los datos:
        Date  Region  OS_Name  Market_Share Device_Category  Is_Active  \
0 2009-01-01  Africa  Windows         97.65         Desktop       True   
1 2009-02-01  Africa  Windows         97.74         Desktop       True   
2 2009-03-01  Africa  Windows         97.58         Desktop       True   
3 2009-04-01  Africa  Windows         97.38         Desktop       True   
4 2009-05-01  Africa  Windows         97.24         Desktop       True   

   Market_Concentration_Index  
0                   9537.1509  
1                   9554.5601  
2                   9523.5485  
3                   9484.8873  
4                   9457.7398  
